# StoreMind AI — retail operations prototype

This adapts the uploaded waste-image classification notebook to a retail scenario. The **runnable demo uses simulated records**, clearly labeled below. It reconciles shelf observations, stock records, and point-of-sale (POS) events into staff tasks. It does not determine theft, change inventory automatically, or contact anyone.

The optional image-training section applies the original ResNet18 transfer-learning approach to three shelf states (`empty`, `low`, `stocked`) once real, labeled shelf photos are available. A shelf-state classifier cannot identify a product or count units by itself. Production use would also need a validated SKU detector/tracker, calibrated cameras, integrations, and privacy review.

In [ ]:
from pathlib import Path
from datetime import timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
print("StoreMind prototype — SIMULATED operational records")

## 1. Simulated inputs

`on_hand` is the inventory system's total count, while `observed_shelf_units` is a camera or staff observation of the visible shelf. They are deliberately separate. `verified_backroom_units` is populated only when staff have checked the backroom. Every observation has a confidence score. No rows below are measurements from a real store.

In [ ]:
as_of = pd.Timestamp("2026-09-29 14:12:00+10:00")
inventory = pd.DataFrame([
    {"sku":"HP-101", "product":"Headphones", "on_hand":4, "verified_backroom_units":pd.NA, "shelf_min":2},
    {"sku":"CB-202", "product":"USB-C cable", "on_hand":0, "verified_backroom_units":0, "shelf_min":3},
    {"sku":"WB-303", "product":"Water bottle", "on_hand":7, "verified_backroom_units":5, "shelf_min":4},
    {"sku":"PH-404", "product":"Phone case", "on_hand":5, "verified_backroom_units":2, "shelf_min":2},
])
observations = pd.DataFrame([
    {"sku":"HP-101", "shelf_id":"A1", "observed_shelf_units":0, "confidence":.94, "observed_at":as_of},
    {"sku":"CB-202", "shelf_id":"B2", "observed_shelf_units":0, "confidence":.97, "observed_at":as_of},
    {"sku":"WB-303", "shelf_id":"C3", "observed_shelf_units":2, "confidence":.93, "observed_at":as_of},
    {"sku":"PH-404", "shelf_id":"D4", "observed_shelf_units":3, "confidence":.91, "observed_at":as_of},
])
movements = pd.DataFrame([
    {"movement_id":"M-001", "sku":"HP-101", "units":1, "at":as_of-pd.Timedelta(minutes=8), "camera_ref":"cam-A1-1410"},
    {"movement_id":"M-002", "sku":"PH-404", "units":1, "at":as_of-pd.Timedelta(minutes=6), "camera_ref":"cam-D4-1406"},
])
pos = pd.DataFrame([
    {"sale_id":"S-001", "sku":"PH-404", "units":1, "at":as_of-pd.Timedelta(minutes=4)},
])
display(inventory)
display(observations)

## 2. Convert shelf observations into staff tasks

A mismatch is a prompt to verify the shelf and backroom, not proof that the inventory count is wrong. Reorder suggestions and restock tasks await staff action.

In [ ]:
def build_shelf_tasks(inventory, observations, confidence_min=.85):
    required_i = {"sku", "product", "on_hand", "verified_backroom_units", "shelf_min"}
    required_o = {"sku", "shelf_id", "observed_shelf_units", "confidence", "observed_at"}
    if not required_i.issubset(inventory.columns) or not required_o.issubset(observations.columns):
        raise ValueError("Missing required inventory or observation columns")
    if inventory.sku.duplicated().any() or observations.sku.duplicated().any():
        raise ValueError("Expected one inventory and one latest observation row per SKU")
    merged = observations.merge(inventory, on="sku", how="left", validate="one_to_one", indicator=True)
    if (merged._merge != "both").any():
        raise ValueError("Observation contains an unknown SKU")
    tasks = []
    for row in merged.itertuples(index=False):
        if not 0 <= row.confidence <= 1 or row.observed_shelf_units < 0 or row.on_hand < 0:
            raise ValueError(f"Invalid observation or stock count for {row.sku}")
        backroom = row.verified_backroom_units
        if row.confidence < confidence_min:
            action, reason = "VERIFY_OBSERVATION", "Observation confidence is too low"
        elif row.observed_shelf_units == 0 and row.on_hand == 0:
            action, reason = "PROPOSE_REORDER", "Shelf is empty and system stock is zero"
        elif row.observed_shelf_units < row.shelf_min and pd.notna(backroom) and backroom > 0:
            action, reason = "RESTOCK_FROM_BACKROOM", "Visible shelf is below minimum; backroom stock was verified"
        elif row.observed_shelf_units == 0 and row.on_hand > 0:
            action, reason = "CHECK_SHELF_AND_BACKROOM", "System shows stock but shelf appears empty"
        elif row.observed_shelf_units < row.shelf_min:
            action, reason = "VERIFY_LOW_SHELF", "Visible shelf is below minimum"
        else:
            action, reason = "OK", "No shelf task indicated"
        tasks.append({"sku":row.sku, "product":row.product, "shelf_id":row.shelf_id,
                      "action":action, "reason":reason, "observed_at":row.observed_at,
                      "confidence":row.confidence})
    return pd.DataFrame(tasks)

shelf_tasks = build_shelf_tasks(inventory, observations)
display(shelf_tasks)

## 3. Reconcile a visible movement with POS

A matched sale explains a movement within the chosen time window. An unmatched movement becomes a **human review item**; it is not labeled theft. Real deployments need accurate timestamps, returns/transfers handling, camera retention rules, and review of false positives.

In [ ]:
def review_movements(movements, pos, window_minutes=30):
    required_m = {"movement_id", "sku", "units", "at", "camera_ref"}
    required_p = {"sale_id", "sku", "units", "at"}
    if not required_m.issubset(movements.columns) or not required_p.issubset(pos.columns):
        raise ValueError("Missing required movement or POS columns")
    if window_minutes <= 0:
        raise ValueError("window_minutes must be positive")
    window = pd.Timedelta(minutes=window_minutes)
    used_sales = set()
    reviews = []
    for m in movements.sort_values("at").itertuples(index=False):
        if m.units <= 0:
            raise ValueError("Movement units must be positive")
        candidates = pos.loc[(pos["sku"] == m.sku) & (pos["units"] >= m.units) &
                             (pos["at"] >= m.at) & (pos["at"] <= m.at + window) &
                             (~pos["sale_id"].isin(used_sales))].sort_values("at")
        sale_id = candidates.iloc[0].sale_id if len(candidates) else None
        if sale_id is not None:
            used_sales.add(sale_id)
        reviews.append({"movement_id":m.movement_id, "sku":m.sku,
                        "status":"MATCHED_POS" if sale_id else "REVIEW_UNMATCHED_MOVEMENT",
                        "sale_id":sale_id, "camera_ref":m.camera_ref, "at":m.at,
                        "next_step":"No incident task" if sale_id else "Staff verify POS, transfers, returns, and footage"})
    return pd.DataFrame(reviews)

movement_reviews = review_movements(movements, pos)
display(movement_reviews)

## 4. Task queue and simple dashboard

Counts below describe only the four **simulated** SKUs and two simulated movements. They are not model accuracy or business-impact estimates.

In [ ]:
staff_queue = shelf_tasks.loc[shelf_tasks.action != "OK", ["sku", "product", "shelf_id", "action", "reason"]].copy()
incident_queue = movement_reviews.loc[movement_reviews.status == "REVIEW_UNMATCHED_MOVEMENT", ["sku", "camera_ref", "status", "next_step"]].copy()
print("SIMULATED staff queue")
display(staff_queue)
print("SIMULATED movement review queue")
display(incident_queue)
fig, ax = plt.subplots(figsize=(9, 3.8))
counts = shelf_tasks.action.value_counts().sort_values()
colors = ["#3D8F9D" if a == "OK" else "#F0A44B" for a in counts.index]
ax.barh(counts.index.str.replace("_", " "), counts.values, color=colors)
ax.set(xlabel="Number of simulated SKUs", title="Shelf task distribution — sample data")
ax.set_xlim(0, max(counts.values) + .5)
for i, v in enumerate(counts.values): ax.text(v + .03, i, str(v), va="center")
fig.tight_layout()
plt.show()

## 5. Optional real image pipeline

Create a CSV manifest with `image_path, sku, shelf_id, capture_session, label`; labels must be `empty`, `low`, or `stocked`. `capture_session` should identify a photo session/camera/time group so near-duplicate frames stay together when splitting train and validation data. A classifier estimates shelf state only; a separate validated system must establish SKU and count. No model metrics are reported until real data are trained and tested.

In [ ]:
SHELF_LABELS = ("empty", "low", "stocked")

def validate_manifest(csv_path):
    from PIL import Image
    df = pd.read_csv(csv_path)
    expected = {"image_path", "sku", "shelf_id", "capture_session", "label"}
    if not expected.issubset(df.columns):
        raise ValueError(f"Manifest needs columns: {sorted(expected)}")
    if len(df) == 0 or df[list(expected)].isna().any().any():
        raise ValueError("Manifest is empty or contains missing required values")
    if not set(df.label).issubset(SHELF_LABELS):
        raise ValueError(f"Labels must be in {SHELF_LABELS}")
    base = Path(csv_path).resolve().parent
    for raw in df.image_path:
        path = Path(raw)
        path = path if path.is_absolute() else base / path
        if not path.is_file(): raise FileNotFoundError(path)
        with Image.open(path) as img: img.verify()
    if df.capture_session.nunique() < 2:
        raise ValueError("At least two capture sessions are needed for a grouped split")
    return df

manifest_path = Path("shelf_images/manifest.csv")
if manifest_path.exists():
    real_manifest = validate_manifest(manifest_path)
    print(f"Validated {len(real_manifest)} real labeled shelf photos")
else:
    print("No real shelf-image manifest supplied; training is skipped.")

### ResNet18 transfer-learning recipe

This mirrors the uploaded notebook's pretrained-image-model approach, with a three-class output. Install PyTorch and torchvision in the notebook environment before calling it. The function uses grouped validation and reports a confusion matrix and macro F1 on held-out capture sessions. **Do not treat validation as a final test**; reserve independent store/camera sessions for final evaluation.

In [ ]:
def train_shelf_state_model(manifest_csv, epochs=3, batch_size=16, seed=42):
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from torchvision import models, transforms
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.metrics import classification_report, confusion_matrix
    from PIL import Image

    frame = validate_manifest(manifest_csv).reset_index(drop=True)
    split = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed)
    train_idx, val_idx = next(split.split(frame, frame.label, groups=frame.capture_session))
    train_frame, val_frame = frame.iloc[train_idx], frame.iloc[val_idx]
    if set(train_frame.label) != set(SHELF_LABELS):
        raise ValueError("Training split must contain all three classes; collect more sessions")
    root = Path(manifest_csv).resolve().parent
    weights = models.ResNet18_Weights.DEFAULT
    preprocess = weights.transforms()
    augment = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(.75, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[.485,.456,.406], std=[.229,.224,.225]),
    ])
    class ShelfDataset(Dataset):
        def __init__(self, rows, transform): self.rows, self.transform = rows.reset_index(drop=True), transform
        def __len__(self): return len(self.rows)
        def __getitem__(self, idx):
            row = self.rows.iloc[idx]
            p = Path(row.image_path)
            if not p.is_absolute(): p = root / p
            with Image.open(p) as image: x = self.transform(image.convert("RGB"))
            return x, SHELF_LABELS.index(row.label)
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet18(weights=weights)
    for p in model.parameters(): p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, len(SHELF_LABELS))
    model.to(device)
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(ShelfDataset(train_frame, augment), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(ShelfDataset(val_frame, preprocess), batch_size=batch_size)
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
        print(f"Epoch {epoch + 1}/{epochs}: training loss {total_loss / len(train_frame):.3f}")
    model.eval()
    truth, predicted = [], []
    with torch.no_grad():
        for x, y in val_loader:
            pred = model(x.to(device)).argmax(1).cpu().tolist()
            predicted.extend(pred)
            truth.extend(y.tolist())
    print(classification_report(truth, predicted, labels=range(3), target_names=SHELF_LABELS, zero_division=0))
    print("Confusion matrix (true rows, predicted columns):")
    print(confusion_matrix(truth, predicted, labels=range(3)))
    return model

# Once real data and dependencies are ready:
# model = train_shelf_state_model("shelf_images/manifest.csv")

## 6. Optional footage preservation for staff review

If a real video file and event offset are available, this helper extracts a short clip. It never uploads or sends it. A `camera_ref` string in the simulation is only a placeholder, not actual evidence. Set access controls and retention policy before use with real footage.

In [ ]:
def preserve_review_clip(video_path, event_offset_seconds, output_path, before=10, after=15):
    import shutil
    import subprocess
    source, target = Path(video_path), Path(output_path)
    if not source.is_file(): raise FileNotFoundError(source)
    if shutil.which("ffmpeg") is None: raise RuntimeError("ffmpeg is required")
    if event_offset_seconds < 0 or before < 0 or after <= 0:
        raise ValueError("Invalid clip time range")
    start = max(0, float(event_offset_seconds) - before)
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", str(start),
                    "-i", str(source), "-t", str(before + after), "-c:v", "libx264",
                    "-c:a", "aac", str(target)], check=True)
    return target

# Example with a real local file only:
# clip = preserve_review_clip("camera_A1.mp4", 132, "review/M-001.mp4")

## 7. Human review and checks

A reviewer can record a finding after checking the source systems. The code does not infer guilt from an unmatched event.

In [ ]:
def record_review(movement_id, reviewer, resolution, note=""):
    allowed = {"sale_found", "staff_transfer", "return_or_restock", "unclear", "other"}
    if not reviewer.strip() or resolution not in allowed:
        raise ValueError(f"Reviewer and one of {sorted(allowed)} are required")
    return {"movement_id":movement_id, "reviewer":reviewer, "resolution":resolution,
            "note":note, "reviewed_at":pd.Timestamp.now(tz="UTC")}

assert shelf_tasks.set_index("sku").loc["HP-101", "action"] == "CHECK_SHELF_AND_BACKROOM"
assert shelf_tasks.set_index("sku").loc["CB-202", "action"] == "PROPOSE_REORDER"
assert shelf_tasks.set_index("sku").loc["WB-303", "action"] == "RESTOCK_FROM_BACKROOM"
assert movement_reviews.set_index("movement_id").loc["M-002", "status"] == "MATCHED_POS"
assert movement_reviews.set_index("movement_id").loc["M-001", "status"] == "REVIEW_UNMATCHED_MOVEMENT"
assert "theft" not in " ".join(movement_reviews.status.str.lower())
print("Prototype checks passed. Replace sample records and validate on real data before operational use.")

## What to connect next

1. Collect consented/lawful, labeled shelf photos across stores, cameras, lighting, and time; reserve independent sessions for testing.
2. Add a SKU-aware detector/counting pipeline with confidence and calibration, then compare visible shelf count with verified backroom and total on-hand counts.
3. Connect read-only POS/inventory/returns streams with reconciled timestamps and event IDs; design queue ownership, evidence retention, and reviewer audit logs.
4. Measure shelf-alert precision and recall, staff workload, time to restock, and false incident-review rate on real pilot data. The four sample SKUs above cannot support performance claims.